## Neural Network accuracy : 86.67


In [42]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1_l2
import matplotlib.pyplot as plt

In [31]:
np.random.seed(42)
tf.random.set_seed(42)



mandatory if we are reading from csv with link

In [43]:
url = "http://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
cols = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "target"
]
df = pd.read_csv(url, names=cols, na_values="?")
df = df.dropna()
df["target"] = df["target"].apply(lambda x: 1 if x > 0 else 0)

print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['target'].value_counts()}")


Dataset shape: (297, 14)
Class distribution:
target
0    160
1    137
Name: count, dtype: int64


In [44]:
X = df.drop("target", axis=1)
y = df["target"]

# Use larger validation set for more stable validation metrics
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [45]:
model = Sequential([
    Dense(64, activation='relu', kernel_initializer='he_normal',
          kernel_regularizer=l1_l2(l1=0.0001, l2=0.001),
          input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu', kernel_initializer='he_normal',
          kernel_regularizer=l1_l2(l1=0.0001, l2=0.001)),
    BatchNormalization(),
    Dropout(0.2),

    Dense(16, activation='relu', kernel_initializer='he_normal',
          kernel_regularizer=l1_l2(l1=0.0001, l2=0.001)),
    Dropout(0.1),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


model building

In [47]:
early_stop = EarlyStopping(
    monitor='val_auc',
    patience=50,  # Much more patience
    restore_best_weights=True,
    mode='max',
    verbose=1,
    min_delta=0.001  # Only stop if no improvement by at least 0.1%
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_auc',
    factor=0.7,
    patience=20,
    min_lr=1e-7,
    mode='max',
    verbose=1
)

lr_scheduler = LearningRateScheduler(lr_schedule, verbose=0)


Model training

In [54]:
print("\n🚀 Starting training with improved strategy...")

history = model.fit(
    X_train, y_train,
    validation_split=0.25,  # Larger validation set
    epochs=240,  # Even more epochs allowed
    batch_size=32,  # Larger batch for stability
    callbacks=[early_stop, reduce_lr, lr_scheduler],
    class_weight=class_weight_dict,
    verbose=1
)


🚀 Starting training with improved strategy...
Epoch 1/240
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8860 - auc: 0.9585 - loss: 0.5153 - precision: 0.8817 - recall: 0.8702 - val_accuracy: 0.8036 - val_auc: 0.8193 - val_loss: 0.9241 - val_precision: 0.8333 - val_recall: 0.7407 - learning_rate: 1.0000e-04
Epoch 2/240
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8968 - auc: 0.9666 - loss: 0.4967 - precision: 0.9053 - recall: 0.8719 - val_accuracy: 0.8036 - val_auc: 0.8212 - val_loss: 0.9241 - val_precision: 0.8333 - val_recall: 0.7407 - learning_rate: 1.9000e-04
Epoch 3/240
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.8953 - auc: 0.9663 - loss: 0.4932 - precision: 0.8985 - recall: 0.8742 - val_accuracy: 0.8036 - val_auc: 0.8206 - val_loss: 0.9256 - val_precision: 0.8333 - val_recall: 0.7407 - learning_rate: 2.8000e-04
Epoch 4/240
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9042 - auc: 0.9676 - loss: 0.4972 - precision: 0.9107 - recall: 0.8789 - val_accuracy

In [57]:
results = model.evaluate(X_test, y_test, verbose=0)
loss, acc, auc, precision, recall = results

print(f"\n✅ Test Results:")
print(f"   Accuracy:  {acc * 100:.2f}%")
print(f"   AUC:       {auc:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1-Score:  {2 * (precision * recall) / (precision + recall):.4f}")
print(f"   Loss:      {loss:.4f}")



✅ Test Results:
   Accuracy:  86.67%
   AUC:       0.9586
   Precision: 0.8788
   Recall:    0.8286
   F1-Score:  0.8529
   Loss:      0.5299


In [58]:
y_pred = (model.predict(X_test, verbose=0) > 0.5).astype(int)
y_pred_proba = model.predict(X_test, verbose=0)

from sklearn.metrics import classification_report, confusion_matrix
print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

print("\n🔍 Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\nTrue Negatives:  {cm[0][0]}")
print(f"False Positives: {cm[0][1]}")
print(f"False Negatives: {cm[1][0]}")
print(f"True Positives:  {cm[1][1]}")



📊 Classification Report:
              precision    recall  f1-score   support

  No Disease       0.86      0.90      0.88        40
     Disease       0.88      0.83      0.85        35

    accuracy                           0.87        75
   macro avg       0.87      0.86      0.87        75
weighted avg       0.87      0.87      0.87        75


🔍 Confusion Matrix:
[[36  4]
 [ 6 29]]

True Negatives:  36
False Positives: 4
False Negatives: 6
True Positives:  29


In [59]:
fig = plt.figure(figsize=(20, 12))


<Figure size 2000x1200 with 0 Axes>